# 07 · Train your first tiny 3D U-Net

    **Audience:** first-year biology students  
    **Time:** about 100 minutes  
    **Prerequisites:** Tutorials 04–06; CPU works, GPU optional  
    **Data:** generated locally; no downloads or personal/clinical data.

    ## By the end, you can

- create paired 3D images and masks without downloading data
- trace `(N,C,D,H,W)` tensors through Conv3d, pooling and skip connections
- train a small 3D U-Net and evaluate held-out Dice

    **How to work:** Run cells from top to bottom (Shift+Enter). Inspect every figure, try the **Your turn** prompts, then compare with the supplied self-checks. A GPU is not necessary unless noted. If you restart the runtime, rerun all cells.

## 1 · Learning objective: segment synthetic 3D nuclei
Instead of starting with gigabyte-sized CT data, generate 3D ellipsoids. Their masks are known exactly, so we can debug the data and evaluation first. We use **different random seeds** for training and validation. Synthetic success is not evidence of performance on real microscopy or medical scans.

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
rng=np.random.default_rng(70)
torch.manual_seed(7)
torch.set_num_threads(min(2,torch.get_num_threads()))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SIZE=24

def make_3d(n,seed,size=SIZE):
    local=np.random.default_rng(seed)
    z,y,x=np.mgrid[:size,:size,:size]
    images,labels=[],[]
    for _ in range(n):
        cz,cy,cx=local.uniform(size*.35,size*.65,3)
        rz,ry,rx=local.uniform(size*.17,size*.28,3)
        r2=((z-cz)/rz)**2+((y-cy)/ry)**2+((x-cx)/rx)**2
        label=(r2<=1).astype(np.float32)
        image=np.clip(0.10+0.9*np.exp(-1.4*r2)+local.normal(0,0.07,(size,size,size)),0,1).astype(np.float32)
        images.append(image[None]);labels.append(label[None])
    return torch.from_numpy(np.stack(images)),torch.from_numpy(np.stack(labels))

train_x,train_y=make_3d(24,701)
val_x,val_y=make_3d(6,702)
print('Training image shape:',tuple(train_x.shape),'Device:',DEVICE)
assert train_x.shape==(24,1,SIZE,SIZE,SIZE)

## 2 · Show an example before training
Do not train any model before inspecting the inputs and labels. Each slice below cuts through the same 3D synthetic object at its center.

In [ ]:
i=0; zmid=SIZE//2
fig,axes=plt.subplots(1,2,figsize=(7,3))
axes[0].imshow(train_x[i,0,zmid],cmap='gray');axes[0].set_title('Synthetic intensity')
axes[1].imshow(train_y[i,0,zmid],cmap='gray');axes[1].set_title('Known mask')
plt.tight_layout();plt.show()

## 3 · Build a minimal 3D U-Net
`Conv3d` replaces 2D filters with cubic filters; `MaxPool3d` compresses all three spatial dimensions. A skip connection preserves fine structure. The final `1×1×1` convolution emits **one foreground logit per voxel**.

In [ ]:
class TinyUNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.Sequential(nn.Conv3d(1,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,4,3,padding=1),nn.ReLU())
        self.pool=nn.MaxPool3d(2)
        self.bridge=nn.Sequential(nn.Conv3d(4,8,3,padding=1),nn.ReLU(),nn.Conv3d(8,8,3,padding=1),nn.ReLU())
        self.up=nn.ConvTranspose3d(8,4,kernel_size=2,stride=2)
        self.dec=nn.Sequential(nn.Conv3d(8,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,4,3,padding=1),nn.ReLU())
        self.head=nn.Conv3d(4,1,1)
    def forward(self,x):
        skip=self.enc(x)
        deep=self.bridge(self.pool(skip))
        restored=self.up(deep)
        joined=torch.cat([restored,skip],dim=1)
        return self.head(self.dec(joined))

model=TinyUNet3D().to(DEVICE)
with torch.no_grad():
    output=model(train_x[:1].to(DEVICE))
print('Model output:',tuple(output.shape))
print('Trainable parameters:',sum(p.numel() for p in model.parameters()))
assert output.shape==train_y[:1].shape

## 4 · Train on small volumes
Foreground occupies only a fraction of voxels. Binary cross-entropy plus soft Dice loss gives a stronger learning signal than accuracy alone. We set a small `EPOCHS` value for CPU use; longer training may help. Training and evaluation use separate volumes.

In [ ]:
def soft_dice_loss(logits,labels):
    p=torch.sigmoid(logits)
    axes=(1,2,3,4)
    return 1-((2*(p*labels).sum(axes)+1)/(p.sum(axes)+labels.sum(axes)+1)).mean()

def total_loss(logits,labels):
    return F.binary_cross_entropy_with_logits(logits,labels)+soft_dice_loss(logits,labels)

optimizer=torch.optim.Adam(model.parameters(),lr=0.008)
EPOCHS=12
loss_trace=[]
for ep in range(EPOCHS):
    model.train(); epoch_losses=[]
    for ids in torch.randperm(len(train_x)).split(4):
        xbatch=train_x[ids].to(DEVICE); ybatch=train_y[ids].to(DEVICE)
        optimizer.zero_grad()
        loss=total_loss(model(xbatch),ybatch)
        loss.backward(); optimizer.step()
        epoch_losses.append(loss.item())
    loss_trace.append(float(np.mean(epoch_losses)))
    if ep in (0,EPOCHS-1):
        print(f'Epoch {ep+1:02d}/{EPOCHS} loss: {loss_trace[-1]:.3f}')
plt.plot(loss_trace);plt.xlabel('Epoch');plt.ylabel('Training loss');plt.show()

## 5 · Predict unseen ellipsoids
Dice evaluates voxel overlap after applying a probability threshold of 0.5. Inspect predictions even if the numeric score looks good.

In [ ]:
def dice_binary(pred,truth):
    pred=pred.bool();truth=truth.bool()
    denom=int(pred.sum()+truth.sum())
    return 1.0 if denom==0 else float(2*(pred&truth).sum()/denom)

model.eval()
with torch.no_grad():
    probabilities=torch.sigmoid(model(val_x.to(DEVICE))).cpu()
predicted=probabilities>0.5
scores=[dice_binary(predicted[i],val_y[i]>0.5) for i in range(len(val_x))]
print('Validation Dice, each held-out volume:',np.round(scores,3))
print('Mean validation Dice:',round(float(np.mean(scores)),3))
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,arr,title in zip(axes,[val_x[0,0,zmid],val_y[0,0,zmid],probabilities[0,0,zmid]],['Unseen image','Reference','Prediction probability']):
    ax.imshow(arr,cmap='gray',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.tight_layout();plt.show()

## 6 · Common failure modes
- **Almost all background:** small targets, insufficient training, or a badly chosen threshold.
- **Great training Dice, poor validation Dice:** overfitting or data mismatch.
- **Out-of-memory:** reduce `SIZE`, batch size or feature channels; GPU RAM usage rises quickly in 3D.
- **Axial distortions:** check physical spacing; this synthetic notebook uses unitless isotropic voxels.

## Your turn
1. Count foreground voxels in one label; what percentage of the volume do they occupy?
2. Double the noise standard deviation (`0.07` → `0.14`) and compare validation Dice after retraining from a fresh model.
3. Draw one **XZ** slice of a prediction (fix y instead of z).

**Challenge:** Compare BCE alone with BCE + Dice while keeping the train/validation split unchanged.

In [ ]:
foreground_fraction=float(val_y[0].mean())
assert 0<foreground_fraction<0.5
assert probabilities.shape==val_y.shape
assert all(np.isfinite(scores))
print('Foreground fraction:',round(foreground_fraction,3))
print('Numerical self-checks passed.')

## Next
A research 3D U-Net also needs standardized file loading, transforms, patch handling and robust inference. MONAI provides these tools. The final notebook introduces them, then links to the repository's full CT spleen-segmentation project.